# VinDr-SpineXR 임플란트(고밀도 이물질) 탐지 — YOLO Colab 학습·평가

`scripts/prepare_spinexr.py convert`로 생성한 `data/vindr-yolo/` 데이터셋을 사용해 **Google Colab GPU**에서 YOLO 모델을 학습하고,
가중치(`best.pt`)와 평가 지표(**Precision, Recall, mAP@0.5, 영상당 오검출 수(FP/image)**)를 산출하는 노트북입니다.

### 사전 준비 (로컬에서 실행)
1. `scripts/prepare_spinexr.py plan` 및 `convert` 실행 → `data/vindr-yolo/` 생성
2. `data/vindr-yolo` 폴더를 zip으로 압축:
   ```bash
   cd data && zip -r vindr-yolo.zip vindr-yolo
   ```
3. 압축한 `vindr-yolo.zip`을 본인 **Google Drive (비공개)** `MyDrive/vindr-spinexr/vindr-yolo.zip`에 업로드
4. Colab 상단 메뉴 **런타임 → 런타임 유형 변경 → 하드웨어 가속기: T4 GPU (또는 L4/A100)** 선택

> ⚠️ **데이터 라이선스 주의**: VinDr-SpineXR은 PhysioNet Restricted Health Data License 1.5.0이 적용됩니다.
> Google Drive에 올린 데이터와 zip 파일은 **본인만 접근(비공개)**으로 유지하고, GitHub 저장소에 커밋하거나 외부 공유하지 마세요.
> 본 모델 결과는 교육용이며 실제 진단 용도가 아닙니다.

## 1. GPU 환경 확인 및 패키지 설치

In [ ]:
import subprocess
import sys

# GPU 확인
res = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if res.returncode == 0:
    print(res.stdout)
else:
    print("⚠️ GPU가 감지되지 않았습니다. [런타임 → 런타임 유형 변경 → GPU]를 선택하세요.")

# Ultralytics 및 보조 패키지 설치
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics", "pyyaml", "pandas", "matplotlib"])

import torch
import ultralytics

print(f"PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
print(f"Ultralytics: {ultralytics.__version__}")

## 2. Google Drive 마운트 및 경로·하이퍼파라미터 설정

- 로컬에서 `prepare_spinexr.py convert` 기본 `--max-size 1024`로 변환했다면 `IMGSZ = 640` (빠른 베이스라인) 또는 `IMGSZ = 1024` (작은 금속/클립 정밀 탐지)를 비교 실험할 수 있습니다.
- 학습 완료 후 가중치와 평가 지표는 `DRIVE_SAVE_DIR`에 자동 백업됩니다.

In [ ]:
from pathlib import Path
import shutil
import zipfile

# Google Drive 마운트 (Colab 환경에서만 실행)
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# ======================== 사용자 설정 ========================
# 1) 데이터 경로 (zip 파일 또는 이미 압축 해제된 폴더 중 존재하는 경로 사용)
DRIVE_ZIP_PATH = Path("/content/drive/MyDrive/vindr-spinexr/vindr-yolo.zip")
LOCAL_DATA_DIR = Path("/content/vindr-yolo")  # 로컬 SSD에 풀어야 학습 속도가 빠름

# 2) 결과 백업 경로 (Google Drive)
RUN_NAME = "spinexr_implant_yolo11n_640"
DRIVE_SAVE_DIR = Path("/content/drive/MyDrive/vindr-spinexr/runs") / RUN_NAME

# 3) 학습 하이퍼파라미터
MODEL_NAME = "yolo11n.pt"   # 후보: "yolo11n.pt", "yolo11s.pt", "yolov8n.pt"
EPOCHS = 50
IMGSZ = 640                 # 640 (기본/빠름) 또는 1024 (고해상도 비교 실험)
BATCH = 16                  # OOM 발생 시 8로 줄이기
PATIENCE = 15
SEED = 42

# 4) 평가 설정
EVAL_IOU_THRESH = 0.50
DEFAULT_CONF = 0.25
CONF_SWEEP = [0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]
# =============================================================

# 압축 해제 (이미 풀려 있으면 건너뜀)
if not (LOCAL_DATA_DIR / "images").exists():
    if not DRIVE_ZIP_PATH.exists():
        raise FileNotFoundError(
            f"데이터 zip 파일을 찾을 수 없습니다: {DRIVE_ZIP_PATH}\n"
            "로컬에서 'cd data && zip -r vindr-yolo.zip vindr-yolo' 실행 후 Drive에 업로드하세요."
        )
    extract_root = LOCAL_DATA_DIR.parent
    extract_root.mkdir(parents=True, exist_ok=True)
    print(f"압축 해제 중: {DRIVE_ZIP_PATH} -> {extract_root}")
    with zipfile.ZipFile(DRIVE_ZIP_PATH, "r") as zf:
        zf.extractall(extract_root)

    # zip 내부 최상위 폴더 이름이 vindr-yolo가 아닌 경우 자동 탐색
    if not (LOCAL_DATA_DIR / "images").exists():
        candidates = list(extract_root.rglob("dataset.yaml"))
        if candidates:
            LOCAL_DATA_DIR = candidates[0].parent
            print(f"dataset.yaml 위치 자동 감지: {LOCAL_DATA_DIR}")
        else:
            raise FileNotFoundError("압축 해제 결과에서 dataset.yaml / images 폴더를 찾지 못했습니다.")

print(f"사용할 데이터 경로: {LOCAL_DATA_DIR}")

## 3. `dataset.yaml` Colab 경로 보정 및 데이터셋 분포 점검

`scripts/prepare_spinexr.py convert`가 생성한 `dataset.yaml`에는 로컬 절대 경로(`path: /Users/.../data/vindr-yolo`)가 기록되어 있습니다.
아래 셀은 이를 현재 Colab 경로(`/content/vindr-yolo`)로 자동 보정한 `dataset_colab.yaml`을 만들고, `train / val / test` 분할별 임플란트(양성)·정상(음성) 영상 수와 박스 수를 검증합니다.

In [ ]:
import pandas as pd
import yaml

orig_yaml = LOCAL_DATA_DIR / "dataset.yaml"
if orig_yaml.exists():
    cfg = yaml.safe_load(orig_yaml.read_text(encoding="utf-8"))
else:
    cfg = {
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "names": {0: "implant"},
    }

# Colab 경로로 path 보정
cfg["path"] = LOCAL_DATA_DIR.resolve().as_posix()
COLAB_YAML = LOCAL_DATA_DIR / "dataset_colab.yaml"
COLAB_YAML.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding="utf-8")
print(f"생성 완료: {COLAB_YAML}")
print(COLAB_YAML.read_text(encoding="utf-8"))

# 분할별 이미지·라벨 무결성 및 양성/정상 분포 점검
IMG_EXTS = {".png", ".jpg", ".jpeg"}
summary_rows = []
for split in ("train", "val", "test"):
    img_dir = LOCAL_DATA_DIR / "images" / split
    lbl_dir = LOCAL_DATA_DIR / "labels" / split
    images = sorted(p for p in img_dir.glob("*") if p.suffix.lower() in IMG_EXTS) if img_dir.exists() else []

    n_implant = 0
    n_normal = 0
    n_missing_lbl = 0
    n_boxes = 0
    for img_path in images:
        lbl_path = lbl_dir / f"{img_path.stem}.txt"
        if not lbl_path.exists():
            n_missing_lbl += 1
            continue
        lines = [ln.strip() for ln in lbl_path.read_text(encoding="utf-8").splitlines() if ln.strip()]
        if lines:
            n_implant += 1
            n_boxes += len(lines)
        else:
            n_normal += 1

    summary_rows.append({
        "split": split,
        "total_images": len(images),
        "implant_images": n_implant,
        "normal_images": n_normal,
        "missing_labels": n_missing_lbl,
        "total_boxes": n_boxes,
    })

df_data = pd.DataFrame(summary_rows)
display(df_data)

if df_data.loc[df_data["split"] == "train", "total_images"].iloc[0] == 0:
    raise RuntimeError("train 이미지가 0장입니다. 데이터 압축 경로를 확인하세요.")
if df_data["missing_labels"].sum() > 0:
    print("⚠️ 일부 이미지의 라벨 파일(.txt)이 누락되었습니다. convert 결과를 확인하세요.")

## 4. YOLO 모델 학습

- `fliplr=0.5`: 척추 X-ray 좌우 반전 증강 허용
- `flipud=0.0`: 상하 반전은 해부학적 방향이 뒤집히므로 비활성화
- `hsv_h=0.0, hsv_s=0.0`: 흑백 X-ray이므로 색상/채도 증강은 끄고 밝기(`hsv_v=0.2`)만 약하게 적용

In [ ]:
from ultralytics import YOLO

TRAIN_PROJECT = Path("/content/runs/detect")
model = YOLO(MODEL_NAME)

train_results = model.train(
    data=str(COLAB_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    seed=SEED,
    device=0 if torch.cuda.is_available() else "cpu",
    project=str(TRAIN_PROJECT),
    name=RUN_NAME,
    exist_ok=True,
    # 의료 흑백 X-ray에 맞춘 증강 설정
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.2,
    degrees=5.0,
    fliplr=0.5,
    flipud=0.0,
    plots=True,
    save=True,
)

RUN_DIR = Path(train_results.save_dir)
BEST_WEIGHTS = RUN_DIR / "weights" / "best.pt"
print(f"학습 결과 폴더: {RUN_DIR}")
print(f"최적 가중치 파일: {BEST_WEIGHTS} (exists={BEST_WEIGHTS.exists()})")

## 5. 공식 평가 지표 산출 (`val` & `test` — Precision, Recall, mAP@0.5)

`best.pt`를 불러와 `val`과 `test` 분할에 대해 Ultralytics 표준 평가(`model.val`)를 실행합니다.

In [ ]:
best_model = YOLO(str(BEST_WEIGHTS))

official_metrics = {}
for split in ("val", "test"):
    split_dir = LOCAL_DATA_DIR / "images" / split
    if not split_dir.exists() or not any(split_dir.iterdir()):
        print(f"[{split}] 이미지가 없어 건너뜁니다.")
        continue
    m = best_model.val(
        data=str(COLAB_YAML),
        split=split,
        imgsz=IMGSZ,
        batch=BATCH,
        iou=EVAL_IOU_THRESH,
        device=0 if torch.cuda.is_available() else "cpu",
        project=str(TRAIN_PROJECT),
        name=f"{RUN_NAME}_eval_{split}",
        exist_ok=True,
        plots=True,
    )
    official_metrics[split] = {
        "precision": round(float(m.box.mp), 4),
        "recall": round(float(m.box.mr), 4),
        "mAP50": round(float(m.box.map50), 4),
        "mAP50_95": round(float(m.box.map), 4),
    }

print("\n=== Ultralytics 공식 평가 결과 ===")
display(pd.DataFrame(official_metrics).T)

## 6. 영상당 오검출 수(FP/image) 및 Confidence 임계값별 평가

Ultralytics `val()` 기본 출력에는 **영상당 오검출 수(FP/image)**가 직접 표시되지 않습니다.
아래 코드는 `test`(및 `val`) 전체 영상에 대해 예측 박스와 정답 박스(GT)를 **IoU ≥ 0.50, confidence 내림차순 1:1 매칭**으로 비교하여 다음 지표를 정확히 산출합니다:

- **Precision / Recall / F1** (지정 `conf` 기준)
- **FP/img (전체)**: 전체 평가 영상 1장당 평균 오검출 박스 수 (`FP / 전체 영상 수`)
- **FP/img (정상)**: 정상(`No finding`, GT=0) 영상 1장당 평균 오검출 박스 수
- **정상 오탐률 (Normal FPR)**: 정상 영상 중 1개 이상 오검출이 발생한 영상 비율 (`FP >= 1인 정상 영상 / 전체 정상 영상`)
- **FP/img (임플란트)**: 임플란트(양성) 영상 1장당 평균 오검출 박스 수

In [ ]:
import numpy as np
import cv2


def yolo_txt_to_xyxy(label_path: Path, img_w: int, img_h: int) -> np.ndarray:
    """YOLO 정규화 라벨(.txt)을 픽셀 단위 [x1, y1, x2, y2] 배열로 변환한다."""
    if not label_path.exists():
        return np.zeros((0, 4), dtype=np.float32)
    boxes = []
    for line in label_path.read_text(encoding="utf-8").splitlines():
        parts = line.strip().split()
        if len(parts) < 5:
            continue
        _, cx, cy, w, h = map(float, parts[:5])
        x1 = (cx - w / 2.0) * img_w
        y1 = (cy - h / 2.0) * img_h
        x2 = (cx + w / 2.0) * img_w
        y2 = (cy + h / 2.0) * img_h
        boxes.append([x1, y1, x2, y2])
    return np.array(boxes, dtype=np.float32) if boxes else np.zeros((0, 4), dtype=np.float32)


def compute_iou_matrix(boxes_a: np.ndarray, boxes_b: np.ndarray) -> np.ndarray:
    """두 박스 집합 (N,4), (M,4) 간의 IoU 행렬 (N,M)을 계산한다."""
    if len(boxes_a) == 0 or len(boxes_b) == 0:
        return np.zeros((len(boxes_a), len(boxes_b)), dtype=np.float32)
    x1 = np.maximum(boxes_a[:, None, 0], boxes_b[None, :, 0])
    y1 = np.maximum(boxes_a[:, None, 1], boxes_b[None, :, 1])
    x2 = np.minimum(boxes_a[:, None, 2], boxes_b[None, :, 2])
    y2 = np.minimum(boxes_a[:, None, 3], boxes_b[None, :, 3])
    inter = np.maximum(0.0, x2 - x1) * np.maximum(0.0, y2 - y1)
    area_a = np.maximum(0.0, boxes_a[:, 2] - boxes_a[:, 0]) * np.maximum(0.0, boxes_a[:, 3] - boxes_a[:, 1])
    area_b = np.maximum(0.0, boxes_b[:, 2] - boxes_b[:, 0]) * np.maximum(0.0, boxes_b[:, 3] - boxes_b[:, 1])
    union = area_a[:, None] + area_b[None, :] - inter
    return np.where(union > 0, inter / union, 0.0)


def collect_predictions(model_obj: YOLO, data_dir: Path, split: str, imgsz: int) -> list[dict]:
    """split 전체 영상에 대해 낮은 conf(0.001)로 1회 추론해 GT와 예측 박스를 모은다."""
    img_dir = data_dir / "images" / split
    lbl_dir = data_dir / "labels" / split
    img_paths = sorted(p for p in img_dir.glob("*") if p.suffix.lower() in IMG_EXTS)
    records = []
    for img_path in img_paths:
        res = model_obj.predict(
            source=str(img_path),
            imgsz=imgsz,
            conf=0.001,
            iou=0.50,
            device=0 if torch.cuda.is_available() else "cpu",
            verbose=False,
        )[0]
        img_h, img_w = res.orig_shape
        gt_boxes = yolo_txt_to_xyxy(lbl_dir / f"{img_path.stem}.txt", img_w, img_h)
        if res.boxes is not None and len(res.boxes) > 0:
            pred_boxes = res.boxes.xyxy.cpu().numpy().astype(np.float32)
            pred_confs = res.boxes.conf.cpu().numpy().astype(np.float32)
        else:
            pred_boxes = np.zeros((0, 4), dtype=np.float32)
            pred_confs = np.zeros((0,), dtype=np.float32)
        records.append({
            "image_id": img_path.stem,
            "image_path": img_path,
            "gt_boxes": gt_boxes,
            "pred_boxes": pred_boxes,
            "pred_confs": pred_confs,
        })
    return records


def evaluate_at_conf(records: list[dict], conf_thresh: float, iou_thresh: float = 0.50) -> tuple[dict, list[dict]]:
    """지정 conf/iou 기준으로 TP, FP, FN, Precision, Recall, F1, 영상당 오검출 수를 계산한다."""
    total_tp = total_fp = total_fn = 0
    n_normal = n_normal_fp_boxes = n_normal_fp_imgs = 0
    n_implant = n_implant_fp_boxes = 0
    per_image = []

    for rec in records:
        gt = rec["gt_boxes"]
        keep = rec["pred_confs"] >= conf_thresh
        preds = rec["pred_boxes"][keep]
        confs = rec["pred_confs"][keep]

        # confidence 내림차순 1:1 greedy 매칭
        order = np.argsort(-confs)
        preds = preds[order]
        confs = confs[order]

        matched_gt = set()
        tp = fp = 0
        iou_mat = compute_iou_matrix(preds, gt)
        for pi in range(len(preds)):
            if len(gt) == 0:
                fp += 1
                continue
            best_gi = int(np.argmax(iou_mat[pi]))
            if iou_mat[pi, best_gi] >= iou_thresh and best_gi not in matched_gt:
                matched_gt.add(best_gi)
                tp += 1
            else:
                fp += 1
        fn = len(gt) - len(matched_gt)

        total_tp += tp
        total_fp += fp
        total_fn += fn

        is_normal = len(gt) == 0
        if is_normal:
            n_normal += 1
            n_normal_fp_boxes += fp
            if fp > 0:
                n_normal_fp_imgs += 1
        else:
            n_implant += 1
            n_implant_fp_boxes += fp

        per_image.append({
            "image_id": rec["image_id"],
            "image_path": rec["image_path"],
            "is_normal": is_normal,
            "tp": tp, "fp": fp, "fn": fn,
            "gt_boxes": gt, "pred_boxes": preds, "pred_confs": confs,
        })

    n_total = len(records)
    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    metrics = {
        "conf": conf_thresh,
        "TP": total_tp,
        "FP": total_fp,
        "FN": total_fn,
        "Precision": round(precision, 4),
        "Recall": round(recall, 4),
        "F1": round(f1, 4),
        "FP_per_image_all": round(total_fp / n_total, 4) if n_total else 0.0,
        "FP_per_normal_image": round(n_normal_fp_boxes / n_normal, 4) if n_normal else 0.0,
        "normal_FPR": round(n_normal_fp_imgs / n_normal, 4) if n_normal else 0.0,
        "FP_per_implant_image": round(n_implant_fp_boxes / n_implant, 4) if n_implant else 0.0,
    }
    return metrics, per_image


# test 분할 기준 임계값별 평가 (test가 없으면 val 사용)
TARGET_SPLIT = "test" if (LOCAL_DATA_DIR / "images" / "test").exists() and any((LOCAL_DATA_DIR / "images" / "test").iterdir()) else "val"
print(f"[{TARGET_SPLIT}] 분할 전체 추론 및 오검출 분석 시작...")
target_records = collect_predictions(best_model, LOCAL_DATA_DIR, TARGET_SPLIT, IMGSZ)

sweep_rows = []
per_image_at_default = []
for c in CONF_SWEEP:
    row, per_img = evaluate_at_conf(target_records, conf_thresh=c, iou_thresh=EVAL_IOU_THRESH)
    sweep_rows.append(row)
    if abs(c - DEFAULT_CONF) < 1e-9:
        per_image_at_default = per_img

df_sweep = pd.DataFrame(sweep_rows)
print(f"\n=== [{TARGET_SPLIT}] Confidence 임계값별 평가 (IoU >= {EVAL_IOU_THRESH}) ===")
display(df_sweep)

best_row = df_sweep.loc[df_sweep["F1"].idxmax()].to_dict()
default_row = df_sweep.loc[(df_sweep["conf"] - DEFAULT_CONF).abs().idxmin()].to_dict()
print(f"\n[요약 - {TARGET_SPLIT}]")
print(f"- 공식 mAP@0.5: {official_metrics.get(TARGET_SPLIT, {}).get('mAP50')}")
print(f"- 기본 conf={DEFAULT_CONF}: Precision={default_row['Precision']}, Recall={default_row['Recall']}, "
      f"영상당 오검출 수(전체)={default_row['FP_per_image_all']}, 정상 영상당 오검출 수={default_row['FP_per_normal_image']} (정상 오탐률={default_row['normal_FPR']:.1%})")
print(f"- 최고 F1 conf={best_row['conf']}: Precision={best_row['Precision']}, Recall={best_row['Recall']}, "
      f"F1={best_row['F1']}, 영상당 오검출 수(전체)={best_row['FP_per_image_all']}, 정상 영상당 오검출 수={best_row['FP_per_normal_image']}")

## 7. 정성 평가 및 오류 분석 시각화 (성공 / 미검출 FN / 정상 오검출 FP)

초록색 박스는 정답(GT), 빨간색 박스는 모델 예측(Pred + confidence)입니다. 생성된 `error_analysis.png`는 보고서에 바로 첨부할 수 있습니다.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches


def plot_cases(cases: list[dict], title: str, save_path: Path | None = None):
    if not cases:
        print(f"{title}: 해당 사례가 없습니다.")
        return
    n = len(cases)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))
    if n == 1:
        axes = [axes]
    for ax, item in zip(axes, cases):
        img = cv2.imread(str(item["image_path"]), cv2.IMREAD_GRAYSCALE)
        ax.imshow(img, cmap="gray")
        for x1, y1, x2, y2 in item["gt_boxes"]:
            ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor="lime", facecolor="none"))
        for (x1, y1, x2, y2), cf in zip(item["pred_boxes"], item["pred_confs"]):
            ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor="red", facecolor="none"))
            ax.text(x1, max(0, y1 - 4), f"{cf:.2f}", color="yellow", fontsize=9, bbox=dict(facecolor="red", alpha=0.6, pad=1))
        ax.set_title(f"{item['image_id'][:10]}...\nTP={item['tp']} FP={item['fp']} FN={item['fn']}", fontsize=10)
        ax.axis("off")
    fig.suptitle(title + " (Green=GT, Red=Pred)", fontsize=13)
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


if per_image_at_default:
    tp_cases = [x for x in per_image_at_default if not x["is_normal"] and x["tp"] > 0 and x["fn"] == 0][:4]
    fn_cases = [x for x in per_image_at_default if x["fn"] > 0][:4]
    fp_normal_cases = [x for x in per_image_at_default if x["is_normal"] and x["fp"] > 0][:4]

    plot_cases(tp_cases, f"Implant TP Success (conf={DEFAULT_CONF})", RUN_DIR / "viz_tp_success.png")
    plot_cases(fn_cases, f"Implant Missed FN (conf={DEFAULT_CONF})", RUN_DIR / "viz_fn_missed.png")
    plot_cases(fp_normal_cases, f"Normal False Positive (conf={DEFAULT_CONF})", RUN_DIR / "viz_fp_normal.png")

## 8. 가중치(`best.pt`) 및 평가 지표 JSON/CSV 저장 (Google Drive 백업 + 로컬 다운로드)

In [ ]:
import json
from datetime import datetime

summary_payload = {
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "run_name": RUN_NAME,
    "model": MODEL_NAME,
    "epochs": EPOCHS,
    "imgsz": IMGSZ,
    "batch": BATCH,
    "seed": SEED,
    "dataset_summary": summary_rows,
    "official_ultralytics_metrics": official_metrics,
    "target_split_for_fp_analysis": TARGET_SPLIT,
    "default_conf_metrics": default_row,
    "best_f1_conf_metrics": best_row,
}

metrics_json_path = RUN_DIR / "metrics_summary.json"
sweep_csv_path = RUN_DIR / "threshold_sweep.csv"
metrics_json_path.write_text(json.dumps(summary_payload, indent=2, ensure_ascii=False), encoding="utf-8")
df_sweep.to_csv(sweep_csv_path, index=False, encoding="utf-8-sig")

print(f"저장 완료: {metrics_json_path}")
print(f"저장 완료: {sweep_csv_path}")

# Google Drive에 주요 결과물 복사
if IN_COLAB:
    DRIVE_SAVE_DIR.mkdir(parents=True, exist_ok=True)
    files_to_backup = [
        BEST_WEIGHTS,
        RUN_DIR / "weights" / "last.pt",
        RUN_DIR / "results.csv",
        RUN_DIR / "results.png",
        metrics_json_path,
        sweep_csv_path,
        RUN_DIR / "viz_tp_success.png",
        RUN_DIR / "viz_fn_missed.png",
        RUN_DIR / "viz_fp_normal.png",
    ]
    for src_f in files_to_backup:
        if src_f.exists():
            shutil.copy2(src_f, DRIVE_SAVE_DIR / src_f.name)
    print(f"\n✅ Google Drive 백업 완료: {DRIVE_SAVE_DIR}")
    print("백업된 파일 목록:", [p.name for p in sorted(DRIVE_SAVE_DIR.iterdir())])

print("\n=== 최종 JSON 요약 ===")
print(metrics_json_path.read_text(encoding="utf-8"))

In [ ]:
# (선택) 브라우저에서 바로 best.pt와 평가 지표 파일 다운로드
if IN_COLAB:
    from google.colab import files
    for p in (BEST_WEIGHTS, metrics_json_path, sweep_csv_path):
        if p.exists():
            files.download(str(p))